In [ ]:
import os
import random

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [2]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-12-04 09:30:45.830075-05:00


#### Imports, data load, 70/30 split, preprocessing

In [ ]:
df_train = pd.read_sql("SELECT * FROM mart.provider_train", con=engine)
df_pred = pd.read_sql("SELECT * FROM mart.provider_test",  con=engine)

id_col = [c for c in df_train.columns if c.lower() in (
    "provider", "provider_id", "npi", "prov_id")][0]
y_col = "label_provider_fraud_1_0"

X_full = df_train.drop(columns=[id_col, y_col]).apply(
    pd.to_numeric, errors="coerce")
y_full = df_train[y_col].astype(int)

X_train, X_hold, y_train, y_hold, id_train, id_hold = train_test_split(
    X_full, y_full, df_train[id_col],
    test_size=0.30, stratify=y_full, random_state=42
)

split_df = pd.DataFrame({id_col: df_train[id_col], "split_flag": "train"})
split_df.loc[split_df[id_col].isin(id_hold), "split_flag"] = "holdout"

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS mart.provider_train_split"))
    split_df.to_sql("provider_train_split", con=conn,
                    schema="mart", index=False)

print(split_df["split_flag"].value_counts())

imp = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imp.fit_transform(
    X_train), columns=X_train.columns, index=X_train.index)
X_hold_imp = pd.DataFrame(imp.transform(
    X_hold),  columns=X_train.columns, index=X_hold.index)

scaler = StandardScaler()
X_train_std = pd.DataFrame(scaler.fit_transform(
    X_train_imp), columns=X_train.columns, index=X_train.index)
X_hold_std = pd.DataFrame(scaler.transform(
    X_hold_imp),  columns=X_train.columns, index=X_hold.index)

X_pred = df_pred.drop(columns=[id_col]).apply(pd.to_numeric, errors="coerce")
X_pred_imp = pd.DataFrame(imp.transform(
    X_pred), columns=X_train.columns, index=X_pred.index)
X_pred_std = pd.DataFrame(scaler.transform(
    X_pred_imp), columns=X_train.columns, index=X_pred.index)

pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
spw = neg / max(1, pos)
print({"train_pos": pos, "train_neg": neg, "neg_to_pos": spw})

split_flag
train      3787
holdout    1623
Name: count, dtype: int64
{'train_pos': 354, 'train_neg': 3433, 'neg_to_pos': 9.69774011299435}


#### Helpers: metrics + CV tuning + CatBoost CV

In [ ]:
def eval_metrics(y_true, y_proba, thresh=0.5):
    y_pred = (y_proba >= thresh).astype(int)
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "pr_auc":    average_precision_score(y_true, y_proba),
        "roc_auc":   roc_auc_score(y_true, y_proba),
    }

def cv_tune_model(model_name, build_model_fn, param_grid, X_tr, y_tr, n_splits=5, n_trials=30):
    """
    Random search over param_grid; score by mean PR-AUC across folds.
    build_model_fn(params) -> estimator with fit / predict_proba.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    def sample_params():
        return {k: random.choice(v) for k, v in param_grid.items()}

    best_score, best_params = -np.inf, None

    for t in range(n_trials):
        params = sample_params()
        aps = []
        for tr_idx, va_idx in skf.split(X_tr, y_tr):
            X_tr_cv, X_va_cv = X_tr.iloc[tr_idx], X_tr.iloc[va_idx]
            y_tr_cv, y_va_cv = y_tr.iloc[tr_idx], y_tr.iloc[va_idx]
            model = build_model_fn(params)
            model.fit(X_tr_cv, y_tr_cv)
            proba = model.predict_proba(X_va_cv)[:, 1]
            aps.append(average_precision_score(y_va_cv, proba))
        mean_ap = float(np.mean(aps))
        if mean_ap > best_score:
            best_score, best_params = mean_ap, params

    print(f"[{model_name}] best PR-AUC (CV): {best_score:.4f}, params: {best_params}")
    best_model = build_model_fn(best_params)
    best_model.fit(X_tr, y_tr)
    return best_model, best_params, best_score

def cv_tune_cat(X_tr, y_tr, grid, n_splits=5, n_trials=30):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    def sample_params():
        return {k: random.choice(v) for k, v in grid.items()}

    best_score, best_params = -np.inf, None
    for t in range(n_trials):
        params = sample_params()
        aps = []
        for tr_idx, va_idx in skf.split(X_tr, y_tr):
            X_tr_cv, X_va_cv = X_tr.iloc[tr_idx], X_tr.iloc[va_idx]
            y_tr_cv, y_va_cv = y_tr.iloc[tr_idx], y_tr.iloc[va_idx]
            cb = CatBoostClassifier(
                loss_function="Logloss", eval_metric="PRAUC",
                auto_class_weights="Balanced",
                iterations=2000,
                random_seed=42,
                verbose=False,
                **params
            )
            cb.fit(X_tr_cv, y_tr_cv, eval_set=(
                X_va_cv, y_va_cv), use_best_model=True)
            proba = cb.predict_proba(X_va_cv)[:, 1]
            aps.append(average_precision_score(y_va_cv, proba))
        mean_ap = float(np.mean(aps))
        if mean_ap > best_score:
            best_score, best_params = mean_ap, params

    print(
        f"[CatBoost] best PR-AUC (CV): {best_score:.4f}, params: {best_params}")
    best_cb = CatBoostClassifier(
        loss_function="Logloss", eval_metric="PRAUC",
        auto_class_weights="Balanced",
        iterations=2000,
        random_seed=42,
        verbose=False,
        **best_params
    )
    best_cb.fit(X_tr, y_tr, eval_set=(X_tr, y_tr), use_best_model=True)
    return best_cb, best_params, best_score

#### Train/tune each model, evaluate on 30% holdout

In [ ]:
results = []

lr_grid = {
    "C":        [0.1, 0.5, 1.0, 2.0, 5.0],
    "l1_ratio": [0.2, 0.5, 0.8],
}


def build_lr(params):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1,
        random_state=42,
        **params
    )


lr_model, lr_params, lr_cv_ap = cv_tune_model(
    "LogReg", build_lr, lr_grid, X_train_std, y_train
)
proba_hold_lr = lr_model.predict_proba(X_hold_std)[:, 1]
metrics_lr = eval_metrics(y_hold, proba_hold_lr, thresh=0.5)
metrics_lr["model"] = "LogReg"
results.append(metrics_lr)

rf_grid = {
    "n_estimators":    [500, 1000],
    "max_depth":       [None, 8, 12],
    "min_samples_leaf": [1, 3, 5],
    "max_features":    ["sqrt", 0.5],
}


def build_rf(params):
    return RandomForestClassifier(
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
        **params
    )


rf_model, rf_params, rf_cv_ap = cv_tune_model(
    "RF", build_rf, rf_grid, X_train_imp, y_train
)
proba_hold_rf = rf_model.predict_proba(X_hold_imp)[:, 1]
metrics_rf = eval_metrics(y_hold, proba_hold_rf, thresh=0.5)
metrics_rf["model"] = "RandomForest"
results.append(metrics_rf)

et_grid = {
    "n_estimators":    [500, 1000],
    "max_depth":       [None, 8, 12],
    "min_samples_leaf": [1, 3, 5],
    "max_features":    ["sqrt", 0.5],
}


def build_et(params):
    return ExtraTreesClassifier(
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
        **params
    )


et_model, et_params, et_cv_ap = cv_tune_model(
    "ExtraTrees", build_et, et_grid, X_train_imp, y_train
)
proba_hold_et = et_model.predict_proba(X_hold_imp)[:, 1]
metrics_et = eval_metrics(y_hold, proba_hold_et, thresh=0.5)
metrics_et["model"] = "ExtraTrees"
results.append(metrics_et)

lgb_grid = {
    "learning_rate":     [0.03, 0.05, 0.08],
    "num_leaves":        [31, 63, 127],
    "min_data_in_leaf":  [20, 50, 100],
    "feature_fraction":  [0.6, 0.8, 1.0],
    "bagging_fraction":  [0.6, 0.8, 1.0],
    "lambda_l1":         [0.0, 1.0, 5.0],
    "lambda_l2":         [0.0, 1.0, 5.0],
}


def build_lgb(params):
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=2000,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
        **params
    )


lgb_model, lgb_params, lgb_cv_ap = cv_tune_model(
    "LightGBM", build_lgb, lgb_grid, X_train_imp, y_train
)
proba_hold_lgb = lgb_model.predict_proba(X_hold_imp)[:, 1]
metrics_lgb = eval_metrics(y_hold, proba_hold_lgb, thresh=0.5)
metrics_lgb["model"] = "LightGBM"
results.append(metrics_lgb)

xgb_grid = {
    "learning_rate":    [0.03, 0.05, 0.1],
    "max_depth":        [3, 5, 7],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
    "reg_lambda":       [0.5, 1.0, 3.0],
}


def build_xgb(params):
    return xgb.XGBClassifier(
        objective="binary:logistic",
        n_estimators=2000,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
        eval_metric="aucpr",
        **params
    )


xgb_model, xgb_params, xgb_cv_ap = cv_tune_model(
    "XGBoost", build_xgb, xgb_grid, X_train_imp, y_train
)
proba_hold_xgb = xgb_model.predict_proba(X_hold_imp)[:, 1]
metrics_xgb = eval_metrics(y_hold, proba_hold_xgb, thresh=0.5)
metrics_xgb["model"] = "XGBoost"
results.append(metrics_xgb)

cb_grid = {
    "learning_rate": [0.03, 0.05],
    "depth":         [4, 6, 8],
    "l2_leaf_reg":   [1.0, 3.0, 5.0],
}

cb_model, cb_params, cb_cv_ap = cv_tune_cat(X_train_imp, y_train, cb_grid)
proba_hold_cb = cb_model.predict_proba(X_hold_imp)[:, 1]
metrics_cb = eval_metrics(y_hold, proba_hold_cb, thresh=0.5)
metrics_cb["model"] = "CatBoost"
results.append(metrics_cb)

KeyboardInterrupt: 

In [ ]:
res_df = pd.DataFrame(results)
res_df = res_df[["model", "accuracy", "precision",
                 "recall", "f1", "pr_auc", "roc_auc"]]
print("Holdout performance (30% of provider_train):")
print(res_df.sort_values("pr_auc", ascending=False).to_string(index=False))

best_row = res_df.sort_values("pr_auc", ascending=False).iloc[0]
best_name = best_row["model"]
print(f"\nBest model by PR-AUC on holdout: {best_name}")

model_map = {
    "LogReg":       lr_model,
    "RandomForest": rf_model,
    "ExtraTrees":   et_model,
    "LightGBM":     lgb_model,
    "XGBoost":      xgb_model,
    "CatBoost":     cb_model,
}
best_model = model_map[best_name]

if best_name == "LogReg":
    proba_pred = best_model.predict_proba(X_pred_std)[:, 1]
else:
    proba_pred = best_model.predict_proba(X_pred_imp)[:, 1]

pred_labels = (proba_pred >= 0.5).astype(int)

pred_out = pd.DataFrame({
    id_col: df_pred[id_col],
    "fraud_score": proba_pred,
    "fraud_label_0_1": pred_labels
})

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS test.provider_predictions_final"))
    pred_out.to_sql("provider_predictions_final", con=conn,
                    schema="test", index=False)

KeyError: "None of [Index(['model', 'accuracy', 'precision', 'recall', 'f1', 'pr_auc', 'roc_auc'], dtype='object')] are in the [columns]"

#### Improvements

#### Threshold tuning per model on the 30% holdout

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

def tune_threshold_f1(y_true, y_proba, label):
    prec, rec, thr = precision_recall_curve(y_true, y_proba)
    prec = prec[:-1]
    rec = rec[:-1]
    thr = thr

    f1_vals = 2 * (prec * rec) / (prec + rec + 1e-9)
    best_idx = int(np.argmax(f1_vals))
    best_thr = float(thr[best_idx])

    tuned_metrics = eval_metrics(y_true, y_proba, thresh=best_thr)
    tuned_metrics.update({
        "best_threshold": best_thr,
        "precision_at_best": float(prec[best_idx]),
        "recall_at_best": float(rec[best_idx]),
        "f1_at_best": float(f1_vals[best_idx]),
    })
    print(f"\n[{label}] best F1 threshold = {best_thr:.4f} "
          f"(precision={prec[best_idx]:.3f}, recall={rec[best_idx]:.3f}, f1={f1_vals[best_idx]:.3f})")
    print("  tuned metrics:", {k: round(v, 4) for k, v in tuned_metrics.items()
                               if k not in ("best_threshold", "precision_at_best", "recall_at_best", "f1_at_best")})
    return tuned_metrics


hold_scores = {
    "LogReg":       proba_hold_lr,
    "RandomForest": proba_hold_rf,
    "ExtraTrees":   proba_hold_et,
    "LightGBM":     proba_hold_lgb,
    "XGBoost":      proba_hold_xgb,
    "CatBoost":     proba_hold_cb,
}

tuned_list = []
for name, scores in hold_scores.items():
    tm = tune_threshold_f1(y_hold, scores, name)
    tm["model"] = name
    tuned_list.append(tm)

tuned_df = (pd.DataFrame(tuned_list)
            [["model", "accuracy", "precision", "recall", "f1",
                "pr_auc", "roc_auc",
                "best_threshold", "precision_at_best", "recall_at_best", "f1_at_best"]]
            .sort_values("pr_auc", ascending=False))

print("\nTUNED (per-model) metrics on holdout (sorted by PR-AUC):")
print(tuned_df.to_string(index=False))


[LogReg] best F1 threshold = 0.7465 (precision=0.592, recall=0.743, f1=0.659)
  tuned metrics: {'accuracy': 0.9279, 'precision': 0.5916, 'recall': 0.7434, 'f1': 0.6589, 'pr_auc': 0.7288, 'roc_auc': 0.9432}

[RandomForest] best F1 threshold = 0.6304 (precision=0.617, recall=0.605, f1=0.611)
  tuned metrics: {'accuracy': 0.9279, 'precision': 0.6174, 'recall': 0.6053, 'f1': 0.6113, 'pr_auc': 0.6907, 'roc_auc': 0.9335}

[ExtraTrees] best F1 threshold = 0.5998 (precision=0.526, recall=0.737, f1=0.614)
  tuned metrics: {'accuracy': 0.9131, 'precision': 0.5258, 'recall': 0.7368, 'f1': 0.6137, 'pr_auc': 0.6982, 'roc_auc': 0.9349}

[LightGBM] best F1 threshold = 0.1194 (precision=0.506, recall=0.776, f1=0.613)
  tuned metrics: {'accuracy': 0.9082, 'precision': 0.5064, 'recall': 0.7763, 'f1': 0.613, 'pr_auc': 0.6803, 'roc_auc': 0.9264}

[XGBoost] best F1 threshold = 0.0949 (precision=0.509, recall=0.776, f1=0.615)
  tuned metrics: {'accuracy': 0.9088, 'precision': 0.5086, 'recall': 0.7763, 'f1'

#### Predict on the test dataset with Logistic Regression

In [ ]:
from sqlalchemy import text
import numpy as np
import pandas as pd

best_thr_lr_v1 = 0.7465

proba_pred_lr = lr_model.predict_proba(X_pred_std)[:, 1]

labels_pred_lr = (proba_pred_lr >= best_thr_lr_v1).astype(int)

pred_lr_out = pd.DataFrame({
    id_col: df_pred[id_col],
    "fraud_score_lr": proba_pred_lr,
    "fraud_label_0_1_lr": labels_pred_lr
})

with engine.begin() as conn:
    conn.execute(
        text("DROP TABLE IF EXISTS test.provider_predictions_lr_final"))
    pred_lr_out.to_sql(
        "provider_predictions_lr_final",
        con=conn,
        schema="test",
        index=False)

Saved LR predictions for 1353 providers to test.provider_predictions_lr_final
    provider  fraud_score_lr  fraud_label_0_1_lr
0   PRV51002            0.01                   0
1   PRV51006            0.01                   0
2   PRV51009            0.02                   0
3   PRV51010            0.04                   0
4   PRV51018            0.02                   0
..       ...             ...                 ...
95  PRV51461            0.02                   0
96  PRV51463            0.06                   0
97  PRV51473            0.17                   0
98  PRV51481            0.02                   0
99  PRV51485            0.03                   0

[100 rows x 3 columns]
